# Statistics for Artificial Intelligence

If probability asks *what might happen and with what likelihood*, statistics asks *what did happen in the data we collected, and what can we infer from it*. Every machine learning pipeline begins with data, and statistics provides the tools to understand that data — its center, spread, shape, relationships, and limitations — before any model is trained.

Statistics also governs how we **evaluate** AI systems: accuracy, precision, recall, confidence intervals, and hypothesis tests all come from statistical thinking. A model that scores 95% on training data but 60% on new data has a statistical problem, not just an engineering one. This notebook covers descriptive statistics, correlation, regression, sampling, normalization, evaluation metrics, and the statistical principles that underpin reliable AI development.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Why Statistics Matters in AI

Before training a model, you must understand your data. Before trusting a model's output, you must measure its performance on representative data. Statistics serves both purposes.

**Exploratory data analysis (EDA)** uses descriptive statistics to find missing values, outliers, skewed distributions, and correlations. These discoveries guide feature engineering, data cleaning, and model selection. A column with zero variance cannot help a model. A feature highly correlated with the target is promising. A dataset with 40% missing values requires a strategy.

**Model evaluation** uses statistical metrics to quantify performance. Classification reports precision and recall. Regression reports mean squared error and R². A/B tests determine whether a new model is genuinely better or the improvement is random noise.

**Generalization** — the central goal of machine learning — is a statistical concept. We fit models on a **sample** (training data) and hope they perform on the **population** (all future data). Sampling theory, bias-variance tradeoff, and confidence intervals formalize when that hope is justified.

---

## 2. Descriptive Statistics: Summarizing Data

Descriptive statistics compress a dataset into interpretable numbers. For a collection of values $x_1, x_2, \ldots, x_n$:

### 2.1 Measures of Central Tendency

**Mean (average):** The arithmetic center of the data.

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

The mean is sensitive to outliers. A single extreme value pulls it toward the tail.

**Median:** The middle value when data is sorted. For even $n$, the average of the two middle values. The median is **robust** — unaffected by extreme outliers.

**Mode:** The most frequently occurring value. Useful for categorical data.

### 2.2 Measures of Spread

**Variance:** Average squared deviation from the mean.

$$s^2 = \frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \bar{x})^2$$

We divide by $n - 1$ (Bessel's correction) for an unbiased estimate of population variance from a sample.

**Standard deviation:** $s = \sqrt{s^2}$ — spread in the same units as the data.

**Range:** Maximum minus minimum. Simple but sensitive to outliers.

**Interquartile range (IQR):** $Q_3 - Q_1$ — the spread of the middle 50% of data. Robust to outliers.

In [ ]:
# House price example (in thousands)
prices = np.array([250, 320, 280, 410, 295, 350, 310, 900, 275, 330])

print("House prices (thousands):", prices)
print(f"\n  Mean:   {prices.mean():.1f}")
print(f"  Median: {np.median(prices):.1f}")
print(f"  Std:    {prices.std(ddof=1):.1f}")
print(f"  Min:    {prices.min()},  Max: {prices.max()}")
print(f"  Q1:     {np.percentile(prices, 25):.1f},  Q3: {np.percentile(prices, 75):.1f}")
print(f"  IQR:    {np.percentile(prices, 75) - np.percentile(prices, 25):.1f}")
print("\n  Note: mean (389) is pulled up by outlier 900; median (320) is more representative.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(prices, bins=8, edgecolor="white", color="steelblue")
axes[0].axvline(prices.mean(), color="red", linestyle="--", label=f"Mean={prices.mean():.0f}")
axes[0].axvline(np.median(prices), color="green", linestyle="--", label=f"Median={np.median(prices):.0f}")
axes[0].set_xlabel("Price (thousands)")
axes[0].set_title("Histogram with Mean vs Median")
axes[0].legend()

axes[1].boxplot(prices, vert=True)
axes[1].set_ylabel("Price (thousands)")
axes[1].set_title("Box Plot (shows IQR and outliers)")

plt.tight_layout()
plt.show()

---

## 3. Percentiles and the Five-Number Summary

A **percentile** is the value below which a given percentage of data falls. The 50th percentile is the median. The 25th and 75th percentiles are the first and third quartiles ($Q_1$, $Q_3$).

The **five-number summary** — minimum, $Q_1$, median, $Q_3$, maximum — describes distribution compactly. A **box plot** visualizes this: the box spans the IQR, the line inside is the median, and whiskers extend to reasonable bounds with outliers plotted as points.

In AI, percentiles are used for latency monitoring ("99th percentile response time"), feature scaling decisions, and detecting skewed distributions that might need transformation before modeling.

---

## 4. Shape of Distributions: Skewness and Kurtosis

**Skewness** measures asymmetry. Positive skew means a long right tail (mean > median). Negative skew means a long left tail. Many real-world variables — income, house prices, website traffic — are right-skewed.

**Kurtosis** measures tail heaviness compared to a Gaussian. High kurtosis means more extreme outliers than a normal distribution.

Skewed features can hurt models that assume symmetry (linear regression) or benefit from **log transformation** to approximate normality. Checking shape before modeling is a core EDA step.

In [ ]:
normal_data = np.random.normal(0, 1, 5000)
skewed_data = np.random.exponential(2, 5000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(normal_data, bins=50, density=True, alpha=0.7, color="steelblue")
axes[0].set_title(f"Symmetric (skew={stats.skew(normal_data):.2f})")
axes[1].hist(skewed_data, bins=50, density=True, alpha=0.7, color="coral")
axes[1].set_title(f"Right-skewed (skew={stats.skew(skewed_data):.2f})")
for ax in axes:
    ax.set_xlabel("Value")
plt.tight_layout()
plt.show()

---

## 5. Populations, Samples, and Sampling

The **population** is the complete set of all possible observations. A **sample** is a subset we actually observe. Machine learning almost always works with samples — we train on available data and hope it represents the population.

**Random sampling** gives each population member an equal chance of selection, reducing **selection bias**. **Stratified sampling** ensures subgroups are proportionally represented — important when classes are imbalanced (e.g., 1% fraud in transaction data).

The **Law of Large Numbers** states that sample statistics converge to population parameters as sample size grows. The **Central Limit Theorem** states that sample means tend toward a normal distribution regardless of the underlying population shape — enabling confidence intervals and hypothesis tests.

In AI, **train/validation/test splits** are sampling decisions. A model evaluated only on training data measures **in-sample** performance; test set performance estimates **out-of-sample** generalization.

In [ ]:
# Sample mean converges to population mean as sample size increases
population = np.random.exponential(scale=50, size=100_000)
true_mean = population.mean()

sample_sizes = [10, 50, 100, 500, 1000, 5000]
sample_means = [np.random.choice(population, size=n).mean() for n in sample_sizes]

plt.figure(figsize=(8, 4))
plt.plot(sample_sizes, sample_means, "o-", label="Sample mean")
plt.axhline(true_mean, color="r", linestyle="--", label=f"Population mean = {true_mean:.2f}")
plt.xscale("log")
plt.xlabel("Sample size")
plt.ylabel("Sample mean")
plt.title("Larger Samples Give More Accurate Estimates")
plt.legend()
plt.show()

---

## 6. Covariance and Correlation

### 6.1 Covariance

Covariance measures how two variables move together. For variables $X$ and $Y$ with means $\bar{x}$ and $\bar{y}$:

$$\text{Cov}(X, Y) = \frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})$$

Positive covariance: when $X$ is above average, $Y$ tends to be above average. Negative: they move in opposite directions. Near zero: little linear relationship. Covariance is scale-dependent — hard to interpret the magnitude.

### 6.2 Pearson Correlation Coefficient

Pearson's $r$ normalizes covariance to the range $[-1, 1]$:

$$r = \frac{\text{Cov}(X, Y)}{s_X \cdot s_Y}$$

- $r = 1$: perfect positive linear relationship.
- $r = -1$: perfect negative linear relationship.
- $r = 0$: no linear relationship (nonlinear relationships may still exist).

| $|r|$ range | Interpretation |
|-------------|----------------|
| 0.0 – 0.3 | Weak |
| 0.3 – 0.7 | Moderate |
| 0.7 – 1.0 | Strong |

**Important:** Correlation does not imply causation. Ice cream sales and drowning deaths are correlated because both increase in summer — neither causes the other.

In [ ]:
n = 200
study_hours = np.random.uniform(1, 10, n)
exam_score = 40 + 5 * study_hours + np.random.normal(0, 8, n)
sleep = np.random.uniform(4, 9, n)
noise = np.random.normal(50, 15, n)

r_study = np.corrcoef(study_hours, exam_score)[0, 1]
r_sleep = np.corrcoef(sleep, exam_score)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(study_hours, exam_score, alpha=0.5)
axes[0].set_xlabel("Study hours")
axes[0].set_ylabel("Exam score")
axes[0].set_title(f"Strong correlation: r = {r_study:.3f}")

axes[1].scatter(sleep, noise, alpha=0.5)
axes[1].set_xlabel("Sleep hours")
axes[1].set_ylabel("Random noise")
axes[1].set_title(f"Weak correlation: r = {np.corrcoef(sleep, noise)[0,1]:.3f}")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for multiple features
df = pd.DataFrame({
    "study_hours": study_hours,
    "exam_score": exam_score,
    "sleep": sleep,
})

corr_matrix = df.corr()
print("Correlation matrix:")
print(corr_matrix.round(3))

plt.figure(figsize=(5, 4))
plt.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="Pearson r")
plt.xticks(range(3), corr_matrix.columns, rotation=45)
plt.yticks(range(3), corr_matrix.columns)
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{corr_matrix.iloc[i,j]:.2f}", ha="center", va="center")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### 6.3 Spearman Rank Correlation

Pearson measures **linear** relationships. **Spearman's $\rho$** measures **monotonic** relationships by correlating ranks rather than raw values. It is more robust to outliers and captures nonlinear but consistently increasing or decreasing trends. Use Spearman when data is ordinal or when the relationship is monotonic but not linear.

In [ ]:
x = np.linspace(1, 10, 100)
y = np.log(x) + np.random.normal(0, 0.05, 100)

pearson_r = stats.pearsonr(x, y)[0]
spearman_r = stats.spearmanr(x, y)[0]

plt.scatter(x, y, alpha=0.6)
plt.xlabel("x")
plt.ylabel("log(x) + noise")
plt.title(f"Nonlinear but monotonic: Pearson r={pearson_r:.3f}, Spearman ρ={spearman_r:.3f}")
plt.show()

---

## 7. Linear Regression and Least Squares

Linear regression models the relationship between a dependent variable $y$ and one or more independent variables $x$:

$$y = \beta_0 + \beta_1 x + \epsilon$$

$\beta_0$ is the **intercept**, $\beta_1$ is the **slope**, and $\epsilon$ is random error. **Ordinary Least Squares (OLS)** finds the line that minimizes the sum of squared residuals:

$$\min_{\beta_0, \beta_1} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \min \sum_{i=1}^{n} (y_i - \beta_0 - \beta_1 x_i)^2$$

The closed-form solution for simple regression:

$$\beta_1 = \frac{\text{Cov}(X, Y)}{\text{Var}(X)} = r \cdot \frac{s_Y}{s_X} \qquad \beta_0 = \bar{y} - \beta_1 \bar{x}$$

Linear regression is both a statistical method and the simplest machine learning model. Deep learning extends the same idea — weighted sums of inputs — with nonlinear activations stacked in layers.

In [ ]:
# OLS linear regression by hand
x = study_hours
y = exam_score

beta_1 = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
beta_0 = y.mean() - beta_1 * x.mean()

y_pred = beta_0 + beta_1 * x

print(f"Regression line: score = {beta_0:.2f} + {beta_1:.2f} × study_hours")

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.5, label="Data")
x_line = np.linspace(x.min(), x.max(), 100)
plt.plot(x_line, beta_0 + beta_1 * x_line, "r-", linewidth=2, label="OLS fit")
plt.xlabel("Study hours")
plt.ylabel("Exam score")
plt.title("Linear Regression")
plt.legend()
plt.show()

### 7.1 R² (Coefficient of Determination)

$R^2$ measures the proportion of variance in $y$ explained by the model:

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}$$

$R^2 = 1$: perfect fit. $R^2 = 0$: model is no better than predicting the mean. $R^2$ ranges from 0 to 1 for standard regression (can be negative for worse-than-mean predictions on test data).

In [ ]:
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r_squared = 1 - ss_res / ss_tot

print(f"R² = {r_squared:.4f}")
print(f"Interpretation: {r_squared*100:.1f}% of score variance explained by study hours.")

---

## 8. Feature Scaling and Normalization

Features on different scales can distort distance-based algorithms (KNN, K-means) and slow gradient descent. Common transformations:

**Standardization (Z-score):** Center to zero mean, scale to unit variance.

$$z = \frac{x - \mu}{\sigma}$$

**Min-Max normalization:** Scale to $[0, 1]$.

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Neural networks and models using gradient descent typically benefit from standardized inputs. Tree-based models (Random Forest, XGBoost) are scale-invariant and do not require normalization.

In [ ]:
raw = np.array([2500, 1800, 3200, 950, 4100])  # house areas in sq ft

standardized = (raw - raw.mean()) / raw.std(ddof=1)
minmax = (raw - raw.min()) / (raw.max() - raw.min())

print("Original:     ", raw)
print("Standardized: ", np.round(standardized, 3))
print("Min-Max [0,1]:", np.round(minmax, 3))

---

## 9. Outliers and Z-Scores

An **outlier** is a data point unusually far from the rest. Outliers may be errors (data entry mistakes), rare events (fraud transactions), or genuine extreme values.

The **Z-score** measures how many standard deviations a point is from the mean:

$$z_i = \frac{x_i - \bar{x}}{s}$$

A common rule: $|z| > 3$ suggests a potential outlier. The IQR method flags points below $Q_1 - 1.5 \times \text{IQR}$ or above $Q_3 + 1.5 \times \text{IQR}$.

In AI, outlier handling affects model quality. Removing erroneous outliers helps. Removing rare but valid events (fraud) hurts. The right approach depends on domain knowledge.

In [ ]:
data = np.array([12, 15, 14, 13, 16, 14, 15, 100, 13, 14])
z_scores = (data - data.mean()) / data.std(ddof=1)

print("Values:  ", data)
print("Z-scores:", np.round(z_scores, 2))
print("\nOutliers (|z| > 2):", data[np.abs(z_scores) > 2])

---

## 10. Hypothesis Testing and Statistical Significance

A **hypothesis test** evaluates whether an observed effect is real or likely due to random chance. The framework:

1. **Null hypothesis ($H_0$):** No effect exists (e.g., new model is no better than old).
2. **Alternative hypothesis ($H_1$):** An effect exists.
3. Compute a **test statistic** from data.
4. Calculate a **p-value**: probability of seeing this result (or more extreme) if $H_0$ were true.
5. If p-value < **significance level** $\alpha$ (commonly 0.05), reject $H_0$.

In AI, hypothesis tests appear in **A/B testing** (is the new recommendation algorithm genuinely better?), **feature selection** (is this feature's correlation with the target statistically significant?), and **comparing model performance** across runs.

In [ ]:
# A/B test: does model B have higher accuracy than model A?
np.random.seed(0)
accuracy_A = np.random.binomial(1, 0.82, 500)  # 500 predictions, 82% accuracy
accuracy_B = np.random.binomial(1, 0.85, 500)  # 85% accuracy

mean_A, mean_B = accuracy_A.mean(), accuracy_B.mean()
t_stat, p_value = stats.ttest_ind(accuracy_B, accuracy_A)

print("A/B Test — Model Comparison:")
print(f"  Model A accuracy: {mean_A:.3f}")
print(f"  Model B accuracy: {mean_B:.3f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.6f}")
print(f"\n  At α=0.05: {'Reject H₀ — B is significantly better' if p_value < 0.05 else 'Fail to reject H₀'}")

---

## 11. Confidence Intervals

A **confidence interval** provides a range of plausible values for a population parameter. A 95% confidence interval for the mean means: if we repeated the sampling process many times, 95% of the intervals would contain the true population mean.

For a sample mean with large $n$:

$$\bar{x} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}$$

where $z_{\alpha/2} = 1.96$ for 95% confidence.

In AI, confidence intervals quantify uncertainty in model metrics: "Model accuracy is 87% ± 2%" is more informative than "Model accuracy is 87%." They help decide whether improvements are meaningful or within noise.

In [ ]:
sample = np.random.binomial(1, 0.87, 1000)
n = len(sample)
mean = sample.mean()
se = sample.std(ddof=1) / np.sqrt(n)
ci_low = mean - 1.96 * se
ci_high = mean + 1.96 * se

print(f"Sample accuracy: {mean:.4f}")
print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"We are 95% confident the true accuracy lies in this range.")

---

## 12. Classification Evaluation Metrics

For classification models, a **confusion matrix** summarizes predictions:

|  | Predicted Positive | Predicted Negative |
|--|-------------------|-------------------|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

From this matrix:

**Accuracy:** $\frac{TP + TN}{TP + TN + FP + FN}$ — fraction correct overall. Misleading when classes are imbalanced.

**Precision:** $\frac{TP}{TP + FP}$ — of predicted positives, how many are correct. "When the model says fraud, how often is it right?"

**Recall (Sensitivity):** $\frac{TP}{TP + FN}$ — of actual positives, how many did we catch. "Of all fraud cases, how many did we detect?"

**F1 Score:** $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ — harmonic mean of precision and recall. Balances both when classes are imbalanced.

**Specificity:** $\frac{TN}{TN + FP}$ — of actual negatives, how many correctly identified.

In [ ]:
def classification_metrics(tp, fp, fn, tn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {"accuracy": accuracy, "precision": precision, "recall": recall,
            "f1": f1, "specificity": specificity}

# Fraud detection: 100 fraud (90 caught), 9900 legit (9800 correct, 100 false alarms)
tp, fp, fn, tn = 90, 100, 10, 9800
m = classification_metrics(tp, fp, fn, tn)

print("Fraud detection confusion matrix:")
print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"\n  Accuracy:    {m['accuracy']:.4f}  (looks great but misleading!)")
print(f"  Precision:   {m['precision']:.4f}  (47% of fraud alerts are real)")
print(f"  Recall:      {m['recall']:.4f}  (caught 90% of fraud)")
print(f"  F1:          {m['f1']:.4f}")
print(f"  Specificity: {m['specificity']:.4f}")

---

## 13. Regression Evaluation Metrics

For regression models predicting continuous values:

**Mean Squared Error (MSE):**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Root Mean Squared Error (RMSE):** $\sqrt{\text{MSE}}$ — same units as $y$.

**Mean Absolute Error (MAE):**

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

MAE is more robust to outliers than MSE (no squaring). MSE penalizes large errors more heavily.

In [ ]:
y_true = np.array([100, 200, 150, 300, 250])
y_pred = np.array([110, 180, 160, 280, 270])

mse = np.mean((y_true - y_pred) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_true - y_pred))

print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")

---

## 14. The Bias-Variance Tradeoff

Model prediction error decomposes into three parts:

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

**Bias** is error from overly simplistic assumptions. A linear model fitting curved data has high bias — it **underfits**.

**Variance** is error from sensitivity to training data fluctuations. A deep tree memorizing every training point has high variance — it **overfits**.

As model complexity increases, bias decreases and variance increases. The sweet spot minimizes total error. This tradeoff guides model selection, regularization, and the choice of training data size.

| | High Bias (Underfit) | High Variance (Overfit) |
|--|---------------------|------------------------|
| **Training error** | High | Low |
| **Test error** | High | High |
| **Fix** | More complex model, more features | More data, regularization, simpler model |

In [ ]:
# Polynomial regression: underfit vs good fit vs overfit
np.random.seed(42)
x = np.linspace(0, 1, 20)
y = np.sin(2 * np.pi * x) + np.random.normal(0, 0.15, 20)

x_plot = np.linspace(0, 1, 200)
degrees = [1, 4, 15]
titles = ["Underfit (degree=1)", "Good fit (degree=4)", "Overfit (degree=15)"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, deg, title in zip(axes, degrees, titles):
    coeffs = np.polyfit(x, y, deg)
    y_fit = np.polyval(coeffs, x_plot)
    ax.scatter(x, y, color="steelblue", zorder=3)
    ax.plot(x_plot, y_fit, "r-", linewidth=2)
    ax.set_title(title)
    ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

---

## 15. Train, Validation, and Test Sets

Evaluating a model on the same data used to train it is statistically invalid — the model may have **memorized** rather than **generalized**. The standard split:

- **Training set (60–80%):** Fit model parameters.
- **Validation set (10–20%):** Tune hyperparameters, select models, early stopping.
- **Test set (10–20%):** Final unbiased performance estimate. Touch only once.

**Cross-validation** rotates the validation set across folds, giving more stable estimates from limited data:

$$\text{CV Score} = \frac{1}{k} \sum_{i=1}^{k} \text{Score on fold } i$$

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression

X = study_hours.reshape(-1, 1)
Y = exam_score

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

train_r2 = model.score(X_train, y_train)
test_r2 = model.score(X_test, y_test)
cv_scores = cross_val_score(model, X, Y, cv=5, scoring="r2")

print(f"Train R²: {train_r2:.4f}")
print(f"Test R²:  {test_r2:.4f}")
print(f"5-fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---

## 16. Handling Imbalanced Data

When one class dominates (99% legitimate transactions, 1% fraud), a model predicting "not fraud" always achieves 99% accuracy but is useless. Statistical responses include:

- **Use appropriate metrics:** F1, precision-recall curve, AUC-ROC instead of accuracy.
- **Resampling:** Oversample minority class or undersample majority.
- **Class weights:** Penalize misclassifying minority class more heavily in the loss function.
- **Stratified splitting:** Preserve class proportions in train/test splits.

In [ ]:
# Simulate imbalanced data: 95% class 0, 5% class 1
n = 1000
labels = np.array([0] * 950 + [1] * 50)
np.random.shuffle(labels)

print(f"Class 0: {(labels == 0).sum()} ({(labels == 0).mean():.1%})")
print(f"Class 1: {(labels == 1).sum()} ({(labels == 1).mean():.1%})")
print(f"\nNaive 'always predict 0' accuracy: {(labels == 0).mean():.1%}")
print("This is why accuracy alone fails on imbalanced data.")

---

## 17. Exploratory Data Analysis Workflow

A systematic EDA process before building AI models:

1. **Inspect shape and types** — rows, columns, numeric vs categorical.
2. **Check missing values** — count and percentage per column.
3. **Summary statistics** — mean, median, std, min, max for numerics.
4. **Visualize distributions** — histograms, box plots for each feature.
5. **Correlation analysis** — heatmap, identify highly correlated features.
6. **Target analysis** — class balance for classification, distribution for regression.
7. **Detect outliers** — Z-scores, IQR method.
8. **Document findings** — inform cleaning, feature engineering, and model choice.

In [ ]:
# Quick EDA on a sample dataset
df = pd.DataFrame({
    "age": np.random.randint(18, 65, 200),
    "income": np.random.lognormal(10.5, 0.5, 200),
    "credit_score": np.random.normal(700, 80, 200).astype(int),
    "approved": np.random.choice([0, 1], 200, p=[0.3, 0.7]),
})
df.loc[np.random.choice(200, 10), "income"] = np.nan

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nSummary statistics:")
print(df.describe().round(2))
print("\nClass balance (approved):")
print(df["approved"].value_counts(normalize=True).round(3))

---

## 18. Key Formulas Reference

| Concept | Formula |
|---------|--------|
| Mean | $\bar{x} = \frac{1}{n}\sum x_i$ |
| Variance | $s^2 = \frac{1}{n-1}\sum(x_i - \bar{x})^2$ |
| Z-score | $z = (x - \bar{x}) / s$ |
| Covariance | $\text{Cov}(X,Y) = \frac{1}{n-1}\sum(x_i - \bar{x})(y_i - \bar{y})$ |
| Pearson r | $r = \text{Cov}(X,Y) / (s_X s_Y)$ |
| OLS slope | $\beta_1 = \text{Cov}(X,Y) / \text{Var}(X)$ |
| R² | $1 - \text{SS}_{\text{res}} / \text{SS}_{\text{tot}}$ |
| MSE | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ |
| Precision | $TP / (TP + FP)$ |
| Recall | $TP / (TP + FN)$ |
| F1 | $2PR / (P + R)$ |
| 95% CI | $\bar{x} \pm 1.96 \cdot s / \sqrt{n}$ |

---

## 19. Summary

Statistics transforms raw data into understanding and validates whether AI models actually work. **Descriptive statistics** — mean, median, variance, percentiles — summarize datasets. **Correlation and regression** reveal relationships between variables. **Sampling theory** explains why train/test splits matter. **Hypothesis tests and confidence intervals** distinguish real improvements from noise.

**Evaluation metrics** — accuracy, precision, recall, F1, MSE, R² — quantify model performance with statistical precision. The **bias-variance tradeoff** guides model complexity choices. **Feature scaling**, **outlier detection**, and **imbalanced data handling** are statistical decisions that precede and surround model training.

Together with linear algebra, calculus, and probability, statistics completes the mathematical foundation for artificial intelligence. Probability tells you what might happen. Statistics tells you what the data shows. Calculus tells you how to optimize. Linear algebra tells you how to compute. All four are necessary to build and evaluate AI systems with rigor.